# Temporal stability and history dependence

One locked cross-user split; model seeds 11/23/37 measure optimization variability, not uncertainty across user splits. Shuffle replicates are averaged within each model seed before mean/SD. The REF checkpoint is reused, not retrained or counted as a new replicate. Primary representation is communicated spikes; pre-reset membrane is secondary. Relative10 is an offline, duration-aware reference, not a guaranteed upper bound. No training or probe fitting is performed by this notebook.

In [1]:
from pathlib import Path
import sys
from IPython.display import display
import pandas as pd

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "core_benchmark_v1/protocol.py").is_file()), None)
if REPO is None:
    raise FileNotFoundError("Open this notebook inside the WritingRing repository")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from core_benchmark_v1.analysis import results_root, load_manifest, table, plot_native, plot_probe, plot_readout
ROOT = results_root(REPO)
MANIFEST = load_manifest(ROOT)
display(MANIFEST)


{'aliases': {'D0': 'O0', 'M0': 'O0', 'R0': 'O0', 'REF': 'O0', 'T0': 'O0'},
 'backbone_runs': 27,
 'completed_runs': 33,
 'expected_runs': 33,
 'identity': '85ccaddc963c4502624f02b11ba420bb5c1ac9037abd2715b11b0f52ca175d99',
 'model_seeds': [11, 23, 37],
 'notes': ['Fixed250 shuffle holds partial/padding bins fixed.',
  'Relative10 is an offline duration-aware reference, not an upper bound.',
  'G_order measures decoder-accessible temporal alignment, not pure information or abstraction.'],
 'probe_rows': 2964,
 'profile': 'production',
 'readout_runs': 6,
 'shuffle_averaging': 'replicates averaged within model seed before seed statistics',
 'status': 'PASS',
 'uncertainty_scope': 'model-seed variability on one locked user split, not cross-split uncertainty'}

## Lag similarity

All legal timestep pairs from all samples are used, not only t=0. Zero-norm/constant-vector comparisons are excluded from undefined metrics and their coverage is reported.

In [2]:
lag = table(ROOT, "lag_similarity")
display(lag)

,case,corr_pair_count,correlation,cosine,cosine_pair_count,lag,lag_ms,layer,normalized_l2,pair_count,seed,split,state
0,O0,45594,0.310575,0.351220,45594,1,15.625,L1,1.149757,59126,11,train,spike
1,O0,45907,0.577311,0.601707,45907,2,31.250,L1,0.885635,58545,11,train,spike
2,O0,44644,0.498809,0.528780,44644,4,62.500,L1,0.977413,57383,11,train,spike
3,O0,42258,0.426450,0.460677,42258,8,125.000,L1,1.060127,55059,11,train,spike
4,O0,37996,0.383615,0.419419,37996,16,250.000,L1,1.111991,50411,11,train,spike
...,...,...,...,...,...,...,...,...,...,...,...,...,...
715,D1,8260,0.857875,0.860835,8260,8,125.000,L3,0.604671,10904,37,test,pre_reset
716,D1,7096,0.686474,0.692649,7096,16,250.000,L3,0.904405,9736,37,test,pre_reset
717,D1,4824,0.397452,0.408875,4824,32,500.000,L3,1.258712,7400,37,test,pre_reset
718,D1,2856,0.160351,0.176858,2856,48,750.000,L3,1.474342,5072,37,test,pre_reset


## Common-suffix replay and state reset

Reset synaptic and membrane state immediately before the final H timesteps, for each layer separately and all layers. Compare the same suffix and the same eligible samples. Full-accumulator and suffix-only results are distinct.

In [3]:
history = table(ROOT, "history_reset")
display(history)

,actual_history_ms,case,classes_present,eligible_samples,full_ba,full_ba_drop,full_suffix_ba,history_steps,requested_history_ms,reset_full_ba,reset_layers,reset_suffix_ba,seed,split,suffix_ba_drop,suffix_score_change_l2
0,46.875,O0,12,126,0.550469,0.000000,0.234830,3,50.0,0.550469,L1,0.248683,11,val,-0.013853,12.954083
1,46.875,O0,12,126,0.550469,0.013889,0.234830,3,50.0,0.536580,L2,0.147258,11,val,0.087572,39.619492
2,46.875,O0,12,126,0.550469,0.013889,0.234830,3,50.0,0.536580,all,0.083333,11,val,0.151497,46.510292
3,93.750,O0,12,126,0.550469,-0.010648,0.253806,6,100.0,0.561117,L1,0.330069,11,val,-0.076263,35.749718
4,93.750,O0,12,126,0.550469,0.008929,0.253806,6,100.0,0.541540,L2,0.120076,11,val,0.133730,64.148994
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
205,500.000,D1,12,146,0.439231,0.104649,0.421872,32,500.0,0.334582,all,0.185680,37,test,0.236192,264.206024
206,1000.000,D1,12,111,0.445121,0.030096,0.457026,64,1000.0,0.415025,L1,0.470581,37,test,-0.013555,185.030533
207,1000.000,D1,12,111,0.445121,-0.000460,0.457026,64,1000.0,0.445581,L2,0.485299,37,test,-0.028274,59.687752
208,1000.000,D1,12,111,0.445121,0.008333,0.457026,64,1000.0,0.436788,L3,0.470554,37,test,-0.013528,25.867561


## Interpretation boundary

History dependence and temporal stability do not by themselves establish high-level abstraction. Different H values may have different eligible cohorts; compare each reset to its matched reference.

In [4]:
display(MANIFEST.get("notes", []))

['Fixed250 shuffle holds partial/padding bins fixed.',
 'Relative10 is an offline duration-aware reference, not an upper bound.',
 'G_order measures decoder-accessible temporal alignment, not pure information or abstraction.']